# 12 — Cross-language Round-trip

COGANT's forward pipeline is language-agnostic at the `ProgramGraph` layer, but the *ingest* stage has to be taught each language via a `LanguagePlugin`. For Python it uses the stdlib `ast` module; for JavaScript / TypeScript it uses the bundled tree-sitter plugin.

This notebook demonstrates the **cross-language path**: parse a JavaScript snippet with the tree-sitter plugin, wrap the result in a `ProgramGraph`, run the forward pipeline through to GNN roles, and then discuss how you'd measure round-trip fidelity (ε) across a language boundary.

**Run from the `cogant/` directory:**
```bash
uv run jupyter nbconvert --to notebook --execute docs/notebooks/12_cross_language.ipynb
```

## 1. The tree-sitter JS plugin

`cogant.plugins.js_plugin.JsLanguagePlugin` is the canonical non-Python example. It routes parsing through `cogant.static.treesitter_parser` so you get the same AST structures the main ingest stage would produce.

Tree-sitter grammars are an optional dependency: on a minimal install the plugin will still import, but `HAS_TREESITTER` is `False` and `parse(...)` returns a sentinel envelope. We handle both cases below.

In [ ]:
from __future__ import annotations

from cogant.plugins.js_plugin import JsLanguagePlugin
from cogant.static import treesitter_parser as _ts

plugin = JsLanguagePlugin()
plugin.initialize({})

print(f"Plugin          : {plugin.metadata.name} v{plugin.metadata.version}")
print(f"Supports        : {sorted(plugin.supported_languages)}")
print(f"tree-sitter OK  : {_ts.HAS_TREESITTER}")

## 2. Parse a small JS snippet

We feed the plugin a three-symbol snippet: one `import`, one `class`, and one top-level `function`. If tree-sitter is available the plugin hands back a rich `parsed` envelope; otherwise we still get a valid (but empty-ish) dict that downstream code can reason about.

In [ ]:
JS_SNIPPET = '''
import { compute } from './core.js';

export class SensorReader {
    read() { return compute(42); }
}

export function validateReading(value) {
    return value >= 0 && value < 1000;
}
'''.strip()

ast = plugin.parse(JS_SNIPPET)
print("AST envelope keys :", sorted(ast.keys()))
print("language          :", ast.get("language"))
print("parsed available  :", ast.get("available"))

symbols = plugin.extract_symbols(ast)
imports = plugin.resolve_imports(ast)

print(f"\nSymbols ({len(symbols)}):")
for s in symbols:
    print(f"  - {s.get('kind', '?'):<10} {s.get('name', '?')}")
print(f"\nImports ({len(imports)}):")
for imp in imports:
    print(f"  - {imp}")

## 3. Wrap the result in a `ProgramGraph`

The forward pipeline expects a typed `ProgramGraph`, not a language-specific AST. We build one by hand so the example is reproducible even when tree-sitter isn't installed — if the plugin returned zero symbols (no tree-sitter backend), we fall back to the symbols we *know* the snippet contains.

In [ ]:
from cogant.schemas.core import Edge, EdgeKind, Node, NodeKind
from cogant.schemas.graph import GraphMetadata, ProgramGraph

KIND_MAP = {
    "class":       NodeKind.CLASS,
    "function":    NodeKind.FUNCTION,
    "method":      NodeKind.METHOD,
    "const":       NodeKind.VARIABLE,
}
DEFAULT_KIND = NodeKind.VARIABLE

# Fallback symbols when tree-sitter isn't available. These mirror the
# structure of JS_SNIPPET above.
FALLBACK_SYMBOLS = [
    {"name": "SensorReader",     "kind": "class"},
    {"name": "read",             "kind": "method"},
    {"name": "validateReading",  "kind": "function"},
]
FALLBACK_IMPORTS = ["./core.js"]

if not symbols:
    symbols = FALLBACK_SYMBOLS
    imports = FALLBACK_IMPORTS
    print("(using fallback symbol set — tree-sitter unavailable)")

graph = ProgramGraph(metadata=GraphMetadata(repo_uri="synthetic://js-snippet"))

mod_id = "mod:snippet"
graph.add_node(Node(id=mod_id, kind=NodeKind.MODULE, name="snippet", qualified_name="snippet.js"))

for i, sym in enumerate(symbols):
    nid = f"sym:{i}:{sym['name']}"
    kind = KIND_MAP.get(str(sym.get("kind", "")).lower(), DEFAULT_KIND)
    graph.add_node(
        Node(id=nid, kind=kind, name=sym["name"], qualified_name=f"snippet.{sym['name']}")
    )
    graph.add_edge(
        Edge(id=f"e:contains:{i}", source_id=mod_id, target_id=nid, kind=EdgeKind.CONTAINS)
    )

import re as _re

def _clean_import_target(raw: str) -> str:
    """Pull the quoted module path out of a raw JS import line."""
    if not raw:
        return raw
    m = _re.search(r"['\"]([^'\"]+)['\"]", raw)
    return m.group(1) if m else raw

for i, raw_target in enumerate(imports):
    target = _clean_import_target(raw_target)
    tid = f"ext:{i}:{target}"
    graph.add_node(Node(id=tid, kind=NodeKind.MODULE, name=target, qualified_name=target))
    graph.add_edge(Edge(id=f"e:imports:{i}", source_id=mod_id, target_id=tid, kind=EdgeKind.IMPORTS))

print(f"Graph: {len(graph.nodes)} nodes, {len(graph.edges)} edges")
for n in graph.nodes.values():
    print(f"  {n.id:<28} kind={n.kind.value:<10} name={n.name}")

## 4. Apply a small DSL ruleset

We don't run the full translation engine here (it's biased toward Python idioms). Instead we apply a four-line DSL ruleset that matches the JS patterns we know are in the snippet: `Sensor*` → OBSERVATION, `validate*` → CONSTRAINT. That's enough to show roles flowing through a cross-language path.

In [ ]:
import yaml

from cogant.translate.dsl import compile_ruleset, load_rules_from_dict

RULES_YAML = '''
rules:
  - name: JsSensorClassObservation
    role: OBSERVATION
    confidence: 0.90
    conditions:
      - node_kind: class
        name_pattern: 'Sensor*'

  - name: JsValidateFunctionConstraint
    role: CONSTRAINT
    confidence: 0.85
    conditions:
      - node_kind: function
        name_pattern: 'validate*'

  - name: JsReadMethodObservation
    role: OBSERVATION
    confidence: 0.70
    conditions:
      - node_kind: method
        name_pattern: 'read*'
'''

compiled = compile_ruleset(load_rules_from_dict(yaml.safe_load(RULES_YAML)))

role_assignment: dict[str, tuple[str, float]] = {}
for node in graph.nodes.values():
    for rule in compiled:
        score = rule.match(node, graph)
        if score > 0.0:
            prev = role_assignment.get(node.id)
            if prev is None or score > prev[1]:
                role_assignment[node.id] = (rule.role, score)

from collections import Counter

role_counts = Counter(role for role, _ in role_assignment.values())
print(f"Assigned {len(role_assignment)} of {len(graph.nodes)} nodes")
print("Role distribution:")
for role, count in role_counts.most_common():
    print(f"  {role:<12} {count}")
print()
print("Per-node assignments:")
for nid, (role, score) in role_assignment.items():
    name = graph.nodes[nid].name
    print(f"  {name:<20} -> {role:<12} (score={score:.2f})")

## 5. Cross-language round-trip ε — approach

A full round-trip for JS would look like:

1. **Forward:** `JS source → tree-sitter parse → ProgramGraph → translate → GNN`.
2. **Reverse:** `GNN → planner → synthesizer → Python package` (COGANT's reverse pipeline targets Python today).
3. **Cross-language projection:** `Python package → JS emitter → JS source'`.
4. **ε measurement:** compare the `ProgramGraph` of `JS source` with the `ProgramGraph` of `JS source'` at the *role* level, not at the syntactic level.

ε is then the Jaccard distance between the two role distributions, plus a per-node identity match rate for symbols that survived the round trip. **Zero syntactic fidelity is acceptable** — two different JS source files can express the same Active Inference model. What matters is that roles are preserved.

In [ ]:
# Illustrative ε computation over the assignment we already have.
# In a real round-trip you'd repeat step 4 on the regenerated source.
original_roles = Counter(role for role, _ in role_assignment.values())

# Simulate a 'round-tripped' assignment by dropping one low-confidence
# match — this is the shape of the comparison, not a real measurement.
roundtripped_roles = Counter(
    role for _, (role, score) in role_assignment.items() if score >= 0.80
)

def jaccard(a: Counter, b: Counter) -> float:
    keys = set(a) | set(b)
    if not keys:
        return 1.0
    inter = sum(min(a[k], b[k]) for k in keys)
    union = sum(max(a[k], b[k]) for k in keys)
    return inter / union if union else 1.0

similarity = jaccard(original_roles, roundtripped_roles)
epsilon = 1.0 - similarity

print(f"original roles       : {dict(original_roles)}")
print(f"round-tripped roles  : {dict(roundtripped_roles)}")
print(f"Jaccard similarity   : {similarity:.3f}")
print(f"\u03b5 (role-level distance): {epsilon:.3f}")

## 6. Sanity check

- The graph must contain the module node plus at least the three 'core' symbols.
- At least one node should have been assigned a role (`SensorReader` via the glob rule).
- ε must be a real number in `[0, 1]`.

In [ ]:
assert len(graph.nodes) >= 4, f"expected >=4 nodes, got {len(graph.nodes)}"
assert len(role_assignment) >= 1, "expected at least one role assignment"
assert 0.0 <= epsilon <= 1.0, f"epsilon out of range: {epsilon}"
print("OK — cross-language path wired end-to-end.")

plugin.shutdown()

## 7. Takeaways

- The JS tree-sitter plugin plugs into the same `LanguagePlugin` contract as the toy plugin from notebook 09; its only extra requirement is the optional `cogant[multilang]` dependency group.
- `ProgramGraph` is the lingua franca across languages — once your plugin can produce one, every downstream stage (rules, GNN, reverse, runtime) is available for free.
- **Cross-language ε is measured at the role level, not the syntactic level.** Two legitimate JS files can encode the same Active Inference model; the round-trip is faithful if the role distributions agree.
- Today COGANT's reverse pipeline emits Python packages. A JS (or other-language) emitter is a natural next plugin — it would read the same `ReverseGNNModel` that notebooks 03 and 04 use.